In [1]:
import os
os.chdir("C:\\Users\\zharif\\Documents\\Dataset TA")  # Move to climate-research-workbench root
print(f"CWD: {os.getcwd()}")

CWD: C:\Users\zharif\Documents\Dataset TA


In [2]:
import xarray as xr

ds1 = xr.open_dataset("data/83434a9ea551e316778a67346b400817.grib", engine="cfgrib")
ds2 = xr.open_dataset("data/cd17853bd829c9a0f301d2e041ebdee9.grib", engine="cfgrib")

skipping variable: paramId==228 shortName='tp'
Traceback (most recent call last):
  File "c:\Users\zharif\miniconda3\Lib\site-packages\cfgrib\dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zharif\miniconda3\Lib\site-packages\cfgrib\dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
    ...<2 lines>...
    )
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='time' value=Variable(dimensions=('time',), data=array([1451606400, 1451628000, 1451649600, ..., 1767160800, 1767182400,
       1767204000], shape=(14612,))) new_value=Variable(dimensions=('time',), data=array([1451584800, 1451628000, 1451671200, ..., 1767074400, 1767117600,
       1767160800], shape=(7306,)))
c:\Users\zharif\miniconda3\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-li

In [3]:
ds1

<xarray.Dataset> Size: 8GB
Dimensions:     (time: 14612, latitude: 193, longitude: 241)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 117kB 2016-01-01 ... 2025-12-31T18:00:00
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 2kB 33.0 32.75 32.5 ... -14.5 -14.75 -15.0
  * longitude   (longitude) float64 2kB 90.0 90.25 90.5 ... 149.5 149.8 150.0
    valid_time  (time) datetime64[ns] 117kB ...
Data variables:
    u10         (time, latitude, longitude) float32 3GB ...
    v10         (time, latitude, longitude) float32 3GB ...
    t2m         (time, latitude, longitude) float32 3GB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-03T15:25 GRIB to CDM+CF via cfgrib-0.9.1...

In [4]:
ds2

<xarray.Dataset> Size: 3GB
Dimensions:     (time: 7306, step: 2, latitude: 193, longitude: 241)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 58kB 2015-12-31T18:00:00 ... 2025-12-31...
  * step        (step) timedelta64[ns] 16B 06:00:00 12:00:00
    surface     float64 8B ...
  * latitude    (latitude) float64 2kB 33.0 32.75 32.5 ... -14.5 -14.75 -15.0
  * longitude   (longitude) float64 2kB 90.0 90.25 90.5 ... 149.5 149.8 150.0
    valid_time  (time, step) datetime64[ns] 117kB ...
Data variables:
    tp          (time, step, latitude, longitude) float32 3GB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-03T15:25 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
# Flatten ds2's (time, step) into valid_time with true 6-hourly precipitation
# ERA5 tp accumulations: step=6h is 0-6h, step=12h is 0-12h
# So: first 6h = step_6h, second 6h = step_12h - step_6h

tp = ds2["tp"].load()

tp_6h = tp.sel(step="06:00:00")    # 0-6h accumulation (already 6-hourly)
tp_12h = tp.sel(step="12:00:00")   # 0-12h accumulation
tp_6h_second = tp_12h - tp_6h      # 6h-12h accumulation (true 6-hourly)

# Assign valid_time as coordinate
tp_6h = tp_6h.assign_coords(time=tp_6h["time"] + tp_6h["step"])
tp_6h_second = tp_6h_second.assign_coords(time=tp_6h_second["time"] + tp_12h["step"])

# Drop step/valid_time coords
for v in [tp_6h, tp_6h_second]:
    for c in ["step", "valid_time"]:
        if c in v.coords:
            v = v.drop_vars(c)

tp_6h = tp_6h.drop_vars(["step", "valid_time"], errors="ignore")
tp_6h_second = tp_6h_second.drop_vars(["step", "valid_time"], errors="ignore")

# Concatenate and sort by time
tp_combined = xr.concat([tp_6h, tp_6h_second], dim="time").sortby("time")

# Convert to dataset
ds2_converted = tp_combined.to_dataset(name="total_precipitation_6hr")

# Drop unwanted non-dimension coords
drop_coords = [c for c in ds2_converted.coords if c not in ds2_converted.dims]
ds2_converted = ds2_converted.drop_vars(drop_coords)

ds2_converted

<xarray.Dataset> Size: 680MB
Dimensions:                   (latitude: 193, longitude: 241, time: 3653)
Coordinates:
  * latitude                  (latitude) float64 2kB 33.0 32.75 ... -14.75 -15.0
  * longitude                 (longitude) float64 2kB 90.0 90.25 ... 149.8 150.0
  * time                      (time) datetime64[ns] 29kB 2016-01-01 ... 2025-...
Data variables:
    total_precipitation_24hr  (time, latitude, longitude) float32 680MB 0.0 ....

In [6]:
# Load ds1 into memory (GRIB lazy reads can fail)
ds1_loaded = ds1.load()

# Drop all non-dimension coords from ds1
drop_coords_ds1 = [c for c in ds1_loaded.coords if c not in ds1_loaded.dims]
ds1_clean = ds1_loaded.drop_vars(drop_coords_ds1)

# Merge (both now have the same 14612 6-hourly time steps)
combined = xr.merge([ds1_clean, ds2_converted])

# Save to zarr
combined.to_zarr("data/era5_combined.zarr", mode="w")
print("Saved to data/era5_combined.zarr")
combined

OSError: [Errno 22] Invalid argument